# LoRA Fine-tuning

Fine-tunes `Qwen2.5-1.5B-Instruct` on Nova's 4-class classification task.

**Steps**: install → load data → format → configure LoRA → train → save → evaluate


In [ ]:
# Cell 1: Install LoRA dependencies
!pip install -q peft trl datasets

In [ ]:
# Cell 2: Bootstrap — add project root + load base model + training data
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("lora_finetune.ipynb"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

from llm_parser import LLMParser
from finetune.train_data import RAW_TRAIN_DATA, build_dataset
from collections import Counter
import json

llm = LLMParser()

counts = Counter(json.loads(o).get("type") for _, o in RAW_TRAIN_DATA)
print(f"Total samples: {len(RAW_TRAIN_DATA)}")
print("Class distribution:", dict(counts))

In [ ]:
# Cell 3: Format dataset using the model's chat template
dataset = build_dataset(llm.tokenizer)
print("Dataset:", dataset)
print("\nSample formatted text (first 600 chars):")
print(dataset[0]["text"][:600])

In [ ]:
# Cell 4: Configure LoRA
# r=8, alpha=16 → ~0.44% trainable params (~6.8M of 1.55B)
from peft import LoraConfig, TaskType, get_peft_model
from config import LORA_R, LORA_ALPHA

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    inference_mode=False,
)

lora_model = get_peft_model(llm.model, lora_cfg)
lora_model.print_trainable_parameters()

In [ ]:
# Cell 5: Train
# CPU: ~15-30 min / epoch.  GPU: ~1-3 min / epoch.
import torch
from trl import SFTTrainer, SFTConfig

train_cfg = SFTConfig(
    output_dir="../nova_lora_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    bf16=False,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    max_seq_length=512,
    dataset_text_field="text",
)

trainer = SFTTrainer(model=lora_model, args=train_cfg, train_dataset=dataset)
print("Starting training ...")
trainer.train()
print("Training done.")

In [ ]:
# Cell 6: Save adapter + merge into base model
from config import LORA_ADAPTER_DIR, LORA_MERGED_DIR

# Save LoRA adapter only (~50 MB)
lora_model.save_pretrained(f"../{LORA_ADAPTER_DIR}")
llm.tokenizer.save_pretrained(f"../{LORA_ADAPTER_DIR}")
print(f"Adapter saved to {LORA_ADAPTER_DIR}/")

# Merge and save full model (same size as base, no peft overhead at runtime)
merged = lora_model.merge_and_unload()
merged.save_pretrained(f"../{LORA_MERGED_DIR}")
llm.tokenizer.save_pretrained(f"../{LORA_MERGED_DIR}")
print(f"Merged model saved to {LORA_MERGED_DIR}/")

In [ ]:
# Cell 7: Evaluate fine-tuned model on hard cases
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM
from llm_parser import LLMParser, UNIFIED_SYSTEM_PROMPT
from config import LORA_MERGED_DIR

ft_llm = LLMParser(model_name=f"../{LORA_MERGED_DIR}")

hard_cases = [
    ("Nova, fuck this light.",                           "needs_clarification"),
    ("Nova, make this room lively.",                     "needs_clarification"),
    ("Nova, it is boring in here.",                      "needs_clarification"),
    ("Nova, it's a bit dark.",                           "needs_clarification"),
    ("Nova, can I still eat this dish after one night?", "general_qa"),
    ("How are you today?",                               "invalid"),
]

print(f"{'Input':<52} {'Expected':<22} {'Got':<22} OK?")
print("-" * 105)
for text, expected in hard_cases:
    result, _, _ = ft_llm.parse_unified(text)
    got = result.get("type", "error")
    ok = "✓" if got == expected else "✗"
    print(f"{text:<52} {expected:<22} {got:<22} {ok}")